27 abril 2026

Regina Tamayo León

## **A08 - Bootstrapping + Aggregating**

**Dataset**

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

df = pd.read_excel('Motor Trend Car Road Tests.xlsx')
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [26]:
X = df[['hp', 'qsec']]
X = sm.add_constant(X)
y = df['mpg']

**2) Regresión lineal**

In [27]:
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           4.18e-07
Time:                        17:17:28   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

**2a) Intervalos de confianza**

In [28]:
print(model.conf_int())

               0          1
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770


**3) Bootstrap**

In [29]:
n = len(df)
B = 1000 
betas = []

In [30]:
for i in range(B):
    # bootstrap con reemplazo
    sample = df.sample(n=n, replace=True)
    
    X = sample[['hp', 'qsec']]
    X = sm.add_constant(X)
    y = sample['mpg']
    
    model = sm.OLS(y, X).fit()
    
    betas.append(model.params.values)
betas = pd.DataFrame(betas, columns=['beta0', 'beta1_hp', 'beta2_qsec'])

In [31]:
beta_media = betas.mean()
beta_std = betas.std()
beta_ci = betas.quantile([0.025, 0.975])

In [32]:
print("Media de betas:")
print(beta_media)

print("Desviación estándar:")
print(beta_std)

print("Intervalos de confianza :")
print(beta_ci)

Media de betas:
beta0         50.357360
beta1_hp      -0.088483
beta2_qsec    -0.975091
dtype: float64
Desviación estándar:
beta0         11.012014
beta1_hp       0.017437
beta2_qsec     0.524510
dtype: float64
Intervalos de confianza :
           beta0  beta1_hp  beta2_qsec
0.025  32.006596 -0.124049   -2.175105
0.975  74.047777 -0.057121   -0.124309


In [33]:
lower = beta_media - 2 * beta_std
upper = beta_media + 2 * beta_std

for i, name in enumerate(['b0', 'b1 (hp)', 'b2 (qsec)']):
    print(f"{name}: [{lower[i]:.4f}, {upper[i]:.4f}]")

b0: [28.3333, 72.3814]
b1 (hp): [-0.1234, -0.0536]
b2 (qsec): [-2.0241, 0.0739]


/var/folders/wn/wbrpbsz12dl1ks05bpsj29bw0000gn/T/ipykernel_76546/1590739167.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"{name}: [{lower[i]:.4f}, {upper[i]:.4f}]")


**Aggregating**

In [17]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


y = df['mpg']
X = df.drop(columns=['mpg', 'model'])
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.5, random_state=42)


**Función Train**

In [18]:
def train_ensemble(X_train, y_train, n_models=1000, n_features=3):
    
    ensemble = []
    
    for i in range(n_models):
        
        vars_i = np.random.choice(X_train.columns, size=n_features, replace=False)
        X_train_i = sm.add_constant(X_train[vars_i])
        
        model = sm.OLS(y_train, X_train_i).fit()
        
        ensemble.append({
            "model": model,
            "vars": vars_i
        })
    
    return ensemble


**Función Test**

In [19]:
def predict_ensemble(ensemble, X_new):
    
    preds = []
    
    for item in ensemble:
        model = item["model"]
        vars_i = item["vars"]
        
        X_new_i = sm.add_constant(X_new[vars_i], has_constant='add')
        
        y_hat = model.predict(X_new_i)
        preds.append(y_hat.values)
    
    preds = np.array(preds)
    
    y_hat_mean = preds.mean(axis=0)
    y_hat_std = preds.std(axis=0)
    
    return y_hat_mean, y_hat_std


Entrenar

In [20]:
ensemble = train_ensemble(X_train, y_train)


Predecir en test

In [21]:
y_hat_test, y_hat_std = predict_ensemble(ensemble, X_test)


In [22]:
r2 = r2_score(y_test, y_hat_test)

print("R2 en test:", r2)


R2 en test: 0.78340649439491
